**WEB SCRAPPING** - HOUSING PRICE DATA

In [2]:
import pandas as pd
import numpy as np 
from bs4 import BeautifulSoup
import lxml 
import requests



# OVERVIEW
This part helps us to import libraries that will interact with hhtps requests,parsing HTML,handling data, scrap the browser data.
requests-
used to parse the requests which is a python module use to parse HTTP requests.
os-
used to create directories and interact with System's OS
Beautifulsoup-
used to scrap the data -pulls the data from html,xml files.It creates a parse tree that can be used to extract data in heirarichal and much                 more readble format


In [4]:
City = 'gurgaon'
headers = {
    'authority': 'www.99acres.com',
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'accept-language': 'en-US,en;q=0.9',
    'cache-control': 'no-cache',
    'dnt': '1',
    'pragma': 'no-cache',
    'referer': f'https://www.99acres.com/flats-in-{City}-ffid-page',
    'sec-ch-ua': '"Chromium";v="107", "Not;A=Brand";v="8"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-user': '?1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/527.36 (KHTML, like Gecko) Chrome/107.0.0.0 Safari/537.36',
}

In [5]:
#pagenumber to start from 
start = int(input("Enter the start of Pagenumber:\n"))
end = start+15
pagenumber = start
req = 0
flats = pd.DataFrame()
try:
    while pagenumber<end:
        i=1
        req+=1
        url =  f'https://www.99acres.com/flats-in-{City}-ffid-page-{pageNumber}'
        page = requests.get(url,headers)
        pagesoup = BeautifulSoup(page.content,'html.parser')
        for soup in pagesoup.select_one("div[data-label='SEARCH']").select( 'section[data-hydration-on-demand="true"]'):
            #extract property name and property name
            try:
                property_name = soup.select_one('a.tupleNew__propertyHeading').text.strip()
                #extract link
                link = soup.select_one('a.tupleNew__propertyHeading')['href']
                society = soup.select_one('tupleNew__locationName').text.strip()
            except:
                continue
            #detail page
            page = requests.get(link,headers=headers)
            dpagesoup = BeautifulSoup(page.content,'html.parser')
            req+=1
            #price Range
            try:
                price = dpagesoup.select_one().text.strip()
            except:
                price=""
            #area
            try:
                area = dpagesoup.select_one('#Carpet\\ Area td.labels_bold').text.strip()
            except:
                area =""
            #Area with type
            try:
                areaWithType = dpagesoup.select_one('#Super\\ Built-up\\ Area td.labels_bold').text.strip()
            except:
                areaWithType =""
            #Configuration
            try:
                bedRoom = dpagesoup.select_one('#Layout td.labels_bold').text.strip().split(',')[0].replace(' BHK', '')
            except:
                bedRoom=""
            try:
                bathRoom = dpagesoup.elect_one('#Layout td.labels_bold').text.strip().split(',')[1].replace(' Baths', '').strip()
            except:
                bathRoom=""
            try:
                balcony = dpagesoup.select_one('#No\\. of Balconies td.labels_bold').text.strip()
            except:
                balcony=""
            try:
                additionalRoom = dpagesoup.select_one('#Layout td.labels_bold').text.strip()
            except:
                additionalRoom =""
    
            #Address
            try:
                address = dpagesoup.select_one().text.strip()
            except:
                address = ""
            #floor number
            try:
                floorNum = dpagesoup.select_one('#Floor\\ Number td.labels_bold').text.strip()
            except:
                floorNum=""
            #facing
            try:
                facing = dpagesoup.select_one('#Facing td.labels_bold').text.strip()
            except:
                facing =""
            try:
                agePossession = dpagesoupselect_one('#specsBand .icon_ageForProperty').find_parent('div').find_next_sibling('div').text.strip()
            except:
                agePossession = ""
    
            #Nearby Locations
            try:
                nearbyLocations = dpagesoup.select_one().text.strip()
            except:
                nearbyLocations = ""
                
            #Description
            try:
                description = dpagesoup.select_one().text.strip()
            except:
                description = ""
            #Furnish Details
            try:
                furnishDetails = dpagesoup.select_one('#Furnishing td.labels_bold').text.strip()
            except:
                furnishDetails=""
                
            #features
            if furnishDetails:
                try:
                    features = dpagesoup.select_one().text.strip()
                except:
                    features = ""
            else:
                try:
                    features = dpagesoup.select_one().text.strip()
                except:
                    features =""
                    
            #rating by features
            try:
                rating = dpagesoup.select_one().text.strip()
            except:
                rating=""
    
            #propertyID
            try:
                property_id = dpagesoup.select_one().text.strip()
            except:
                property_id =""           
        
            property_data = {
            'property_name': property_name,
            'link': link,
            'society': society,
            'price': price,
            'area': area,
            'areaWithType': areaWithType,
            'bedRoom': bedRoom,
            'bathRoom': bathRoom,
            'balcony': balcony,
            'additionalRoom': additionalRoom,
            'address': address,
            'floorNum': floorNum,
            'facing': facing,
            'agePossession': agePossession,
            'nearbyLocations': nearbyLocations,
            'description': description,
            'furnishDetails': furnishDetails,
            'features': features,
            'rating': rating,
            'property_id': property_id
        }
            temp_df = pd.DataFrame.from_records([property_data])
            flats = pd.concat([flats,temp_df],ignore_index=True)
            if req%4==0:
                time.sleep(10)
            if req%15==0:
                time.sleep(50)
        pagenumber+=1
            
except AttributeError as e:
    print(e)
    print("----------------")
    print("""Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.\n
            You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.""")
flats.to_csv(mode='a',header=True,index=False)  
    

<>:155: SyntaxWarning: invalid escape sequence '\ '
<>:155: SyntaxWarning: invalid escape sequence '\ '
C:\Users\91959\AppData\Local\Temp\ipykernel_19648\1186104710.py:155: SyntaxWarning: invalid escape sequence '\ '
  You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.""")


Enter the start of Pagenumber:
 100


C:\Users\91959\AppData\Local\Temp\ipykernel_19648\1186104710.py:155: SyntaxWarning: invalid escape sequence '\ '
  You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.""")


NameError: name 'pageNumber' is not defined